# Cuando el camino más corto no es el mejor — Modelado del terreno y función de costo

**Problema:** evacuar una zona inundada hacia un refugio. La ruta más corta puede cruzar zonas de alto riesgo.

**Pregunta eje:** *Si dos rutas llegan al mismo refugio, pero una es más corta y la otra atraviesa zonas de mayor riesgo, ¿cómo determina el sistema cuál es la mejor?*
Comparando el costo acumulado $J(P)=\sum C(u,v)$, no la longitud.

## Función de costo
$$C(u,v)=\begin{cases} d(u,v)\,\bigl(1+\alpha\,R(v)^{\beta}\bigr) & v \text{ transitable}\\ \infty & R(v)=-1\end{cases}$$

- $d(u,v)$: 1 (ortogonal) o $\sqrt2$ (diagonal).
- $R(v)$: nivel de riesgo de la celda destino (0 seco, 1–3 anegación, -1 bloqueo total).
- $\alpha$: aversión al riesgo. $\beta$: exponente no lineal.

| Nivel | Significado | Factor (α=2, β=1.5) |
|---|---|---|
| 0 | Seco | 1.00 |
| 1 | Leve | 3.00 |
| 2 | Moderado | 6.66 |
| 3 | Severo | 11.39 |
| -1 | Bloqueo | ∞ |


In [ ]:
from __future__ import annotations

import math

import numpy as np

Node = tuple[int, int]


class GridEnvironment:
    """Cuadrícula de riesgo de inundación con función de costo de transición.

    Niveles: 0 seco, 1-3 anegación creciente, -1 bloqueo total.
    Costo: C(u, v) = d(u, v) * (1 + alpha * R(v) ** beta); inf si v está bloqueada.
    """

    BLOCKED = -1
    MAX_RISK = 3
    _ORTHO = ((-1, 0), (1, 0), (0, -1), (0, 1))
    _DIAG = ((-1, -1), (-1, 1), (1, -1), (1, 1))

    def __init__(self, rows: int, cols: int, alpha: float = 2.0,
                 beta: float = 1.5, allow_diagonal: bool = False) -> None:
        if rows <= 0 or cols <= 0:
            raise ValueError("rows y cols deben ser positivos")
        if alpha < 0 or beta < 1:
            raise ValueError("se requiere alpha >= 0 y beta >= 1")
        self.rows = rows
        self.cols = cols
        self.alpha = alpha
        self.beta = beta
        self.allow_diagonal = allow_diagonal
        self.grid: np.ndarray = np.zeros((rows, cols), dtype=np.int8)

    # ---- validación ----
    def in_bounds(self, r: int, c: int) -> bool:
        """True si (r, c) está dentro de la cuadrícula."""
        return 0 <= r < self.rows and 0 <= c < self.cols

    def _check_level(self, risk_level: int) -> None:
        if risk_level != self.BLOCKED and not 0 <= risk_level <= self.MAX_RISK:
            raise ValueError(f"risk_level inválido: {risk_level} (use -1 o 0..3)")

    # ---- modificación del terreno ----
    def set_cell(self, r: int, c: int, risk_level: int) -> None:
        """Asigna el nivel de riesgo de una celda."""
        if not self.in_bounds(r, c):
            raise IndexError(f"celda fuera de límites: ({r}, {c})")
        self._check_level(risk_level)
        self.grid[r, c] = risk_level

    def set_region_risk(self, r_start: int, r_end: int, c_start: int,
                        c_end: int, risk_level: int) -> None:
        """Asigna un nivel a la región rectangular [r_start, r_end] x [c_start, c_end] (inclusive)."""
        if r_start > r_end or c_start > c_end:
            raise ValueError("rango de región invertido")
        if not (self.in_bounds(r_start, c_start) and self.in_bounds(r_end, c_end)):
            raise IndexError("región fuera de límites")
        self._check_level(risk_level)
        self.grid[r_start:r_end + 1, c_start:c_end + 1] = risk_level

    def load_matrix(self, matrix: np.ndarray | list[list[int]]) -> None:
        """Carga una matriz 2D completa validando dimensiones y niveles (-1 o 0..3)."""
        arr = np.asarray(matrix, dtype=np.int8)
        if arr.shape != (self.rows, self.cols):
            raise ValueError(f"dimensiones inválidas: se esperaba {(self.rows, self.cols)}, se recibió {arr.shape}")
        if not np.isin(arr, (-1, 0, 1, 2, 3)).all():
            raise ValueError("la matriz contiene niveles fuera de {-1, 0, 1, 2, 3}")
        self.grid[:, :] = arr

    def reset(self) -> None:
        """Limpia el terreno a ceros (todo seco)."""
        self.grid.fill(0)

    # ---- consultas ----
    def is_blocked(self, r: int, c: int) -> bool:
        """True si la celda es intransitable (bloqueada o fuera de límites)."""
        return (not self.in_bounds(r, c)) or self.grid[r, c] == self.BLOCKED

    def cost(self, u: Node, v: Node) -> float:
        """Costo de transición u -> v (v adyacente). inf si v está bloqueada."""
        if self.is_blocked(*v):
            return math.inf
        dr, dc = abs(u[0] - v[0]), abs(u[1] - v[1])
        if max(dr, dc) != 1:
            raise ValueError(f"{u} y {v} no son adyacentes")
        d = math.sqrt(2.0) if dr + dc == 2 else 1.0
        risk = float(self.grid[v])
        return d * (1.0 + self.alpha * risk ** self.beta)

    def get_neighbors(self, node: Node) -> list[tuple[Node, float]]:
        """Vecinos transitables de `node` como ((r, c), costo). Nunca incluye bloqueados."""
        r, c = node
        if not self.in_bounds(r, c):
            raise IndexError(f"nodo fuera de límites: {node}")
        result: list[tuple[Node, float]] = []
        for dr, dc in self._ORTHO:
            nr, nc = r + dr, c + dc
            if not self.is_blocked(nr, nc):
                result.append(((nr, nc), self.cost(node, (nr, nc))))
        if self.allow_diagonal:
            for dr, dc in self._DIAG:
                nr, nc = r + dr, c + dc
                if self.is_blocked(nr, nc):
                    continue
                # Anti corner-cutting: impide diagonal cuando las dos ortogonales intermedias están bloqueadas
                if self.is_blocked(r + dr, c) and self.is_blocked(r, c + dc):
                    continue
                result.append(((nr, nc), self.cost(node, (nr, nc))))
        return result


In [ ]:
# Smoke test: mapa 5x5 con un bloqueo
env = GridEnvironment(5, 5)
env.set_cell(2, 2, -1)
assert env.is_blocked(2, 2)
nb = dict(env.get_neighbors((2, 1)))
assert (2, 2) not in nb and set(nb) == {(1, 1), (3, 1), (2, 0)}, nb
print("Smoke test OK:", nb)

# Costos ponderados (alpha=2, beta=1.5)
env.reset()
env.set_cell(1, 0, 1)
env.set_cell(0, 1, 3)
assert math.isclose(env.cost((0, 0), (1, 0)), 1 + 2 * 1 ** 1.5)      # 3.0
assert math.isclose(env.cost((0, 0), (0, 1)), 1 + 2 * 3 ** 1.5)      # ~11.39
assert env.cost((0, 0), (1, 0)) < env.cost((0, 0), (0, 1))           # más riesgo, más costo
env.set_cell(0, 1, -1)
assert env.cost((0, 0), (0, 1)) == math.inf
assert (0, 1) not in dict(env.get_neighbors((0, 0)))

# Región, carga matricial y reset
env.set_region_risk(0, 1, 0, 1, 2)
assert (env.grid[0:2, 0:2] == 2).all()
env.load_matrix(np.zeros((5, 5), dtype=np.int8))
assert (env.grid == 0).all()
env.reset()
assert (env.grid == 0).all()

# Diagonales: costo sqrt(2) y anti corner-cutting
d = GridEnvironment(3, 3, allow_diagonal=True)
assert math.isclose(dict(d.get_neighbors((1, 1)))[(0, 0)], math.sqrt(2))
d.set_cell(0, 1, -1)
d.set_cell(1, 0, -1)
assert (0, 0) not in dict(d.get_neighbors((1, 1)))   # ambas bloqueadas -> impide corte de esquina
d.set_cell(1, 0, 0)
assert (0, 0) in dict(d.get_neighbors((1, 1)))       # solo una bloqueada -> permitido

# Validaciones
for bad in (lambda: env.set_cell(9, 9, 0), lambda: env.set_cell(0, 0, 7)):
    try:
        bad()
        raise AssertionError("debió fallar")
    except (IndexError, ValueError):
        pass
print("Todas las pruebas pasaron")
